## **Arabic Named Entity Recognition**
Named entity recognition (NER) is the task of finding spans of text that constitute proper names and labeling the type of such entities.
As we discussed in the lecture, this task can be formulated as a sequence labeling problem, e.g., using the BIO tagging scheme.
Your task is to build a NER tagger for Arabic using Conditional Random Fields (CRFs) and explore different sets of features to achieve improvement over the baseline tagger.
Arabic is a morphologically rich language, which poses certain challenges to NLP.
We use ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).
Since ANERcorp only has train and test splits, we take 10% of the train as a development set.


## **The Task**
Your task is to improve the feature function (`word2features()`).
Currently, the baseline model only takes into account previous and next words.
You may use external tools to obtain useful features such as POS and morphological tags.
You will also need to submit a report describing your method, result, and analysis (1-2 pages).

## **Data Preparation**

In [1]:
%pip install scikit-learn seqeval
%pip install python-crfsuite
%pip install camel-tools
!test -f ANERcorp-CamelLabSplits.zip || wget https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip
!unzip -o ANERcorp-CamelLabSplits.zip -x "__MACOSX/*"
# MSA morphological analyzer (calima-msa-r13) + MLE disambiguator, used for POS / morphological features
!camel_data -i disambig-mle-calima-msa-r13

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Archive:  ANERcorp-CamelLabSplits.zip
  inflating: ANERcorp-CamelLabSplits/ANERCorp_Benajiba.txt  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt  
  inflating: ANERcorp-CamelLabSplits/README.txt  
  inflating: ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt  
No new packages will be installed.


In [2]:
from sklearn.model_selection import train_test_split

# read ANER named entity dataset
def read_data(file_path):
    with open(file_path, mode='r') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                # split the line by space
                token = line.strip().split(' ')
                # unpack the token
                word, ner = token
                # append the tuple (word, ner tag) to a list for one sentence
                sent.append((word, ner.replace('PERS', 'PER')))
            # if the line is empty and the list is not empty
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

# read training and test data
_train_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt')
test_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt')

# since this dataset does not have dev set, we take 10% of the train as dev set
train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

## **Feature Engineering**
Here, your task is to design a function that tranforms each word into a set of features.
You may want to use existing tools to get POS, morphological features, etc.
The current model gives `0.70` in $F_1$ on the dev set.

In [3]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        'word': word
    }

    if i > 0:
        prev_word = sent[i-1][0]

        features.update({
            '-1:word': prev_word
        })
    else:
        features['BOS'] = True

    if i < len(sent)-1:
        next_word = sent[i+1][0]

        features.update({
            '+1:word': next_word,
        })
    else:
        features['EOS'] = True

    return features

Extract the features from the data and train your model.

The training/evaluation code is wrapped into reusable functions, so each round only needs to pass a new feature function. Training parameters are kept fixed (`BASE_PARAMS`) while comparing feature sets.

In [4]:
import pycrfsuite
from seqeval.metrics import f1_score, classification_report


def sent2features(sent, feat_fn=word2features):
    return [feat_fn(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

def sent2tokens(sent):
    return [token for token, label in sent]


# training parameters
BASE_PARAMS = {
    'c1': 1.0,   # coefficient for L1 penalty
    'c2': 1e-3,  # coefficient for L2 penalty
    'max_iterations': 50,  # stop earlier

    # include transitions that are possible, but not observed
    'feature.possible_transitions': True
}


def train_crf(feat_fn, params=BASE_PARAMS, model_path='arabic-ner.crfsuite'):
    # create pycrfsuite.Trainer instance and load the training data
    trainer = pycrfsuite.Trainer(verbose=False)
    for sent in train_sents:
        trainer.append(sent2features(sent, feat_fn), sent2labels(sent))
    trainer.set_params(params)
    trainer.train(model_path)
    return model_path


baseline_model = train_crf(word2features, model_path='baseline.crfsuite')

## **Evaluation**

Evaluate your model on the development set and the test set.

**Note**: You may run evaluation on the test set **ONLY ONCE** at the end. You should not use the test set to check how well your model is currently performing and modify the model accordingly.
Use the dev set, or create a tune set for this purpose.

In [5]:
def evaluate(model_path, feat_fn, sents, report=True):
    # instantiate the trained tagger and tag the sentences
    tagger = pycrfsuite.Tagger()
    tagger.open(model_path)
    y_pred = [tagger.tag(sent2features(sent, feat_fn)) for sent in sents]
    y_true = [sent2labels(sent) for sent in sents]
    if report:
        print(classification_report(y_true, y_pred, digits=4))
    return f1_score(y_true, y_pred)


# log of dev results for every round
results = []

def log_result(name, description, f1):
    results.append({'round': name, 'features': description, 'dev_f1': round(f1, 4)})
    print(f'[{name}] dev F1 = {f1:.4f}')


# evaluate the baseline on the dev set
f1 = evaluate(baseline_model, word2features, dev_sents)
log_result('R1', 'baseline: word, -1/+1 word, BOS/EOS', f1)

              precision    recall  f1-score   support

         LOC     0.9423    0.7406    0.8293       397
        MISC     0.8431    0.4943    0.6232        87
         ORG     0.8812    0.5086    0.6449       175
         PER     0.8284    0.4606    0.5920       241

   micro avg     0.8980    0.5967    0.7170       900
   macro avg     0.8737    0.5510    0.6724       900
weighted avg     0.8903    0.5967    0.7100       900

[R1] dev F1 = 0.7170


In [6]:
import pandas as pd
pd.DataFrame(results)

,round,features,dev_f1
0,R1,"baseline: word, -1/+1 word, BOS/EOS",0.717


### Orthographic features

In [7]:
import re

AL_PREFIXES = ('ال', 'وال', 'بال', 'فال', 'كال', 'لل')
LATIN_RE = re.compile(r'[A-Za-z]')

def word2features_orth(sent, i):
    word = sent[i][0]
    features = word2features(sent, i)
    features.update({
        'prefix1': word[:1],
        'prefix2': word[:2],
        'prefix3': word[:3],
        'suffix1': word[-1:],
        'suffix2': word[-2:],
        'suffix3': word[-3:],
        'len': str(min(len(word), 10)),
        'has_al': word.startswith(AL_PREFIXES),
        'is_digit': word.isdigit(),
        'has_digit': any(ch.isdigit() for ch in word),
        'has_latin': bool(LATIN_RE.search(word)),
        'is_punct': all(not ch.isalnum() for ch in word),
    })
    return features


orth_model = train_crf(word2features_orth, model_path='orth.crfsuite')
f1 = evaluate(orth_model, word2features_orth, dev_sents)
log_result('+ orthographic', f1)
pd.DataFrame(results)

              precision    recall  f1-score   support

         LOC     0.9202    0.8136    0.8636       397
        MISC     0.7377    0.5172    0.6081        87
         ORG     0.8425    0.6114    0.7086       175
         PER     0.7990    0.6763    0.7326       241

   micro avg     0.8587    0.7089    0.7766       900
   macro avg     0.8249    0.6547    0.7282       900
weighted avg     0.8550    0.7089    0.7737       900



TypeError: log_result() missing 1 required positional argument: 'f1'